# 草稿生成助手

In [ ]:
# ===== 1. 导入依赖 =====
# os：读取环境变量，例如 OPENAI_API_KEY / OPENAI_BASE_URL。
import os
# Path：用来兼容从项目根目录或 notebook 所在目录启动 Jupyter 的情况。
from pathlib import Path
# Annotated：给类型额外绑定 LangGraph reducer，例如 add_messages。
# Sequence：表示 messages 是一个消息序列。
# TypedDict：定义 LangGraph state 的结构。
from typing import Annotated, Sequence, TypedDict

# load_dotenv：从 .env 文件加载 API Key 等配置。
from dotenv import load_dotenv
# BaseMessage：LangChain 所有消息类型的基类。
# SystemMessage：系统提示词，用来规定模型身份和行为。
# HumanMessage：用户消息。
# AIMessage：模型回复。
# ToolMessage：工具执行结果消息。
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage
# @tool 装饰器：把普通 Python 函数包装成模型可以调用的工具。
from langchain_core.tools import tool
# ChatOpenAI：OpenAI 聊天模型客户端；这里也可以通过 OPENAI_BASE_URL 接入中转服务。
from langchain_openai import ChatOpenAI
# StateGraph：LangGraph 的状态图；END 表示图结束。
from langgraph.graph import StateGraph, END
# add_messages 是 reducer：当节点返回新的 messages 时，它负责把新消息追加/合并到旧消息里。
from langgraph.graph.message import add_messages
# ToolNode：LangGraph 预置节点，专门负责执行 AIMessage 里的 tool_calls。
from langgraph.prebuilt import ToolNode

# ===== 2. 加载 .env =====
# 如果你在 langgraph_quickstart 目录下打开 notebook，优先读当前目录的 .env。
env_path = Path(".env")
# 如果你在项目根目录 /home/lee/langchain_study 打开 notebook，就读取 langgraph_quickstart/.env。
if not env_path.exists():
    env_path = Path("langgraph_quickstart/.env")

# 加载后，代码里就可以通过 os.getenv("OPENAI_API_KEY") 读取 key。
load_dotenv(env_path)


In [ ]:
# ===== 3. 定义全局文档内容 =====
# 这个变量用来模拟“当前正在编辑的文档”。
# update 工具会修改它，save 工具会把它写入本地 txt 文件。
document_content = ""


class AgentState(TypedDict):
    """LangGraph 中流转的状态结构。

    可以把 state 理解成整个图共享的一份“工作台数据”。
    每个节点都会收到 state，也可以返回一部分 state 更新。
    """
    # messages 保存完整对话历史，包括用户消息、模型消息、工具结果消息等。
    #
    # Annotated[..., add_messages] 的意思是：
    # 当某个节点返回 {"messages": [新消息]} 时，不要用新列表直接覆盖旧 messages，
    # 而是使用 add_messages 这个 reducer，把新消息追加/合并到旧 messages 后面。
    #
    # 如果没有 add_messages，节点返回 messages 时通常会替换原来的 messages，
    # 这样对话历史就容易丢失。
    messages: Annotated[Sequence[BaseMessage], add_messages]


@tool
def update(content: str) -> str:
    """使用新的 content 更新当前文档内容。"""
    # global 表示这里要修改外层的 document_content 变量，而不是创建局部变量。
    global document_content
    document_content = content
    # 工具的返回值会被 ToolNode 包装成 ToolMessage，再追加进 messages。
    return "文件内容已更新，当前内容为：" + document_content


@tool
def save(filename: str) -> str:
    """将当前文档内容保存到指定文件。

    Args:
        filename: 要保存的文件名，建议以 .txt 结尾。
    """
    global document_content

    # 简单保护：如果用户没有写 .txt，就提示实际应该使用的文件名。
    # 这里目前只是返回文件名，并没有真正保存；如果希望自动补后缀，
    # 可以改成 filename = f"{filename}.txt"。
    if not filename.endswith(".txt"):
        return f"{filename}.txt"

    try:
        # 把当前 document_content 写入本地文件。
        with open(filename, "w") as file:
            file.write(document_content)
        print(f"文件已保存为 {filename}")
        return f"文件已成功保存为 {filename}"
    except Exception as e:
        # 工具异常不要直接让整个图崩掉，而是把错误信息返回给模型/用户。
        return f"保存文件时出错: {e}"


## `add_messages` 是什么？

`add_messages` 是 LangGraph 给 `messages` 字段使用的 reducer，也就是“状态合并规则”。

如果某个节点返回：

```python
return {"messages": [new_message]}
```

因为 `messages` 被定义成：

```python
messages: Annotated[Sequence[BaseMessage], add_messages]
```

LangGraph 不会用 `[new_message]` 覆盖旧消息，而是会把它追加/合并到已有对话历史后面。这样 agent 才能记住前面用户说过什么、模型回复过什么、工具执行结果是什么。


In [ ]:
# ===== 4. 创建工具和绑定模型 =====
# tools 列表告诉模型：你可以选择调用 update 和 save 这两个工具。
tools = [update, save]

# 创建 ChatOpenAI 模型。
# openai_api_base 从 OPENAI_BASE_URL 读取；如果 .env 没写，就默认使用你的中转地址。
# bind_tools(tools) 会把工具 schema 发给模型，让模型在需要时返回 tool_calls。
model = ChatOpenAI(
    model="gpt-4o",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai-proxy.org/v1"),
).bind_tools(tools)


def our_agent(state: AgentState) -> AgentState:
    """Agent 节点：负责读取消息、调用模型，并返回模型回复。

    在图里，这个函数对应名为 "agent" 的节点。
    每次图运行到 agent 节点时，LangGraph 会把当前 state 传进来。
    这个函数再调用大模型，让模型决定：
    1. 直接回复用户；
    2. 或者生成 tool_calls，让后面的 ToolNode 去执行工具。
    """
    # 系统提示词：规定这个助手的职责和工具使用规则。
    # 它告诉模型：你是文档助手，可以更新文档，可以保存文档，修改后要展示内容。
    system_prompt = SystemMessage(
        content=(
            "你是一个智能助手，帮助用户更新和修改文档。"
            "你可以使用update工具来修改或更新文档内容，使用save工具来保存文档，"
            "并且在修改后总是展示当前文档内容。请根据用户的输入合理使用这些工具。"
        )
    )

    # 如果 messages 为空，说明这是第一次进入 agent 节点。
    # 这里先构造一条引导语，提示用户告诉助手要创建什么文档。
    if not state["messages"]:
        user_input = "我正准备帮你更新文档内容，请告诉我你想要创建什么内容。"
        user_message = HumanMessage(content=user_input)
    else:
        # 如果已经有历史消息，说明前面模型/工具已经运行过一轮。
        # 这里通过 input() 继续向用户提问，形成一个交互式 notebook demo。
        user_input = input("你想要创建一个怎样的文档")
        print(f"用户输入: {user_input}")
        user_message = HumanMessage(content=user_input)

    # 组装发给模型的完整上下文：
    # system_prompt：告诉模型它是谁、能做什么。
    # state["messages"]：之前累积的用户消息、AI 回复、工具结果。
    # user_message：本轮新的用户输入。
    all_messages = [system_prompt] + list(state["messages"]) + [user_message]

    # 调用模型。由于前面 bind_tools(tools)，模型可能返回普通文本，也可能返回 tool_calls。
    response = model.invoke(all_messages)
    print(f"模型回复: {response.content}")

    # 如果模型决定调用工具，tool_calls 会保存在 AIMessage 里。
    # 下一步 ToolNode 会读取这些 tool_calls，并执行对应的 Python 工具函数。
    if hasattr(response, "tool_calls") and response.tool_calls:
        for tool_call in response.tool_calls:
            print(f"工具调用: {tool_call}")

    # 返回本节点新增的消息。
    # 因为 AgentState.messages 使用 add_messages，所以这里返回的消息会被追加到历史 messages。
    # 不需要手动返回完整 state，LangGraph 会根据 reducer 自动合并。
    return {"messages": [user_message, response]}


In [ ]:
# ===== 5. 定义流程控制函数和打印函数 =====
def should_continue(state: AgentState) -> str:
    """判断图是否继续运行。

    这个函数会被 add_conditional_edges 使用。
    它的返回值必须匹配后面映射表里的 key："continue" 或 "end"。
    """
    messages = state["messages"]
    if not messages:
        # 没有消息时，继续回到 agent，让它开始对话。
        return "continue"

    # 从后往前找最近的 AIMessage。
    # 如果模型明确表示文档已经保存，就结束图。
    for message in reversed(messages):
        if (
            isinstance(message, AIMessage)
            and "saved" in message.content.lower()
            and "document" in message.content.lower()
        ):
            return "end"

    # 否则继续运行：tools -> agent -> tools ...
    return "continue"


def print_messages(messages):
    """打印最近几条关键消息，方便观察图的运行过程。"""
    if not messages:
        return

    # 这里为了避免刷屏，只看最后三条消息。
    for message in messages[-3:]:
        # ToolMessage 表示工具执行后的返回结果。
        # 例如 update 工具返回“文件内容已更新...”。
        if isinstance(message, ToolMessage):
            print("TOOL Result:", message.content)


In [ ]:
# ===== 6. 构建 LangGraph 状态图 =====
# StateGraph(AgentState) 表示：这个图运行时共享的状态结构是 AgentState。
graph = StateGraph(AgentState)

# 添加 agent 节点：负责调用模型、生成回复或工具调用请求。
graph.add_node("agent", our_agent)

# 添加 tools 节点：负责真正执行模型提出的 tool_calls。
tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

# 设置入口节点：图启动后先进入 agent。
graph.set_entry_point("agent")

# tools 节点执行完后，根据 should_continue 的结果决定下一步。
# 如果返回 "continue"，就回到 agent 继续对话。
# 如果返回 "end"，就进入 END，结束图。
graph.add_conditional_edges(
    "tools",
    should_continue,
    {
        "continue": "agent",
        "end": END,
    },
)

# agent 运行后，总是进入 tools 节点。
# 如果模型返回了 tool_calls，ToolNode 会执行工具；
# 如果没有工具调用，ToolNode 通常不会产生新的工具结果。
graph.add_edge("agent", "tools")

# compile() 把上面声明的节点和边编译成可运行对象。
app = graph.compile()


In [ ]:
# ===== 7. 可视化图结构 =====
# 这一步会把 LangGraph 的节点和边画出来，方便理解执行流。
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))


In [ ]:
# ===== 8. 运行文档助手 =====
def run_document_agent():
    """启动文档助手，并流式打印每一步状态。"""
    print("\n=====DRAFTER=====")

    # 初始状态：messages 为空，表示还没有任何对话历史。
    state = {"messages": []}

    # stream_mode="values" 表示每一步都返回当前完整 state。
    # 每个 step 类似：{"messages": [...]}。
    for step in app.stream(state, stream_mode="values"):
        if "messages" in step:
            print_messages(step["messages"])

    print("\n=====END=====")


In [ ]:
# 在 notebook 里运行这一格，就会启动交互式文档助手。
# 注意：这个 demo 里使用了 input()，运行过程中会等待你在输入框里回复。
if __name__ == "__main__":
    run_document_agent()
